# Algoritmo de la Multiplicación Sintética

## 1. Introducción y Motivación Pedagógica

Al dictar el curso de **Cálculo Diferencial** en la **Universidad del Quindío** (alrededor de 2018), surgió la necesidad práctica de generar funciones polinómicas aleatorias para los estudiantes, de modo que pudieran practicar la búsqueda de raíces (cortes con el eje $x$) aplicando el algoritmo de la **División Sintética**.

Una primera aproximación consistió en definir parámetros generales. Por ejemplo, para una función cuadrática con raíces $x = a$ y $x = b$:

$$f(x) = (x - a)(x - b) = x^2 - (a + b)x + ab$$

Esta fórmula explícita permite generar funciones cuadráticas aleatorias seleccionando $a, b \in [-n, n]$. Aunque este método funciona muy bien para grado $2$ (y con algo de trabajo algebraico para grado $3$), el problema se vuelve combinatoriamente incómodo cuando queremos generar polinomios de **grado $4$ o superior** mediante expansión simbólica directa.


In [1]:
import random

def formatear_polinomio(coeficientes, variable="x"):
    """Construye una representación algebraica legible a partir de los coeficientes."""
    grado = len(coeficientes) - 1
    terminos = []
    for i, coef in enumerate(coeficientes):
        exp = grado - i
        if coef == 0:
            continue

        signo = "-" if coef < 0 else "+"
        val = abs(coef)

        if exp == 0:
            cuerpo = f"{val}"
        elif exp == 1:
            cuerpo = f"{variable}" if val == 1 else f"{val}{variable}"
        else:
            cuerpo = f"{variable}^{exp}" if val == 1 else f"{val}{variable}^{exp}"

        if not terminos:
            terminos.append(f"-{cuerpo}" if coef < 0 else cuerpo)
        else:
            terminos.append(f" {signo} {cuerpo}")

    return "".join(terminos) if terminos else "0"


def cuadratica_parametrica(a, b):
    """Genera coeficientes de f(x) = (x - a)(x - b) = x^2 - (a + b)x + ab."""
    return [1, -(a + b), a * b]


def cubica_parametrica(a, b, c):
    """Genera coeficientes de f(x) = (x - a)(x - b)(x - c) mediante fórmulas explícitas."""
    return [1, -(a + b + c), (a * b + a * c + b * c), -(a * b * c)]


random.seed(2018)
n = 9
a, b = random.randint(-n, n), random.randint(-n, n)
coefs_2 = cuadratica_parametrica(a, b)
print(f"Grado 2 con raíces a={a}, b={b} en [-{n}, {n}]:")
print(f"  f(x) = {formatear_polinomio(coefs_2)}")

c = random.randint(-n, n)
coefs_3 = cubica_parametrica(a, b, c)
print(f"\nGrado 3 con raíces a={a}, b={b}, c={c}:")
print(f"  f(x) = {formatear_polinomio(coefs_3)}")


Grado 2 con raíces a=8, b=-5 en [-9, 9]:
  f(x) = x^2 - 3x - 40

Grado 3 con raíces a=8, b=-5, c=-9:
  f(x) = x^3 + 6x^2 - 67x - 360


## 2. El Algoritmo de la Multiplicación Sintética

La solución propuesta consiste en aplicar el algoritmo de la **División Sintética al revés**.

Supongamos que deseamos construir el polinomio con raíces $x = 2, 3, 5, 7$:

1. Partimos del coeficiente principal inicial $c^{(0)} = [1]$ (que representa el polinomio constante $1$).
2. De la primera raíz $x = 2$, transponemos el $2$ para formar el factor $(x - 2)$. Para la multiplicación usamos únicamente el número $-2$:
   - Multiplicamos $-2$ por $1$ y ubicamos el resultado en la diagonal hacia abajo: obtenemos $[1, -2]$, correspondiente a $x - 2$.
3. El siguiente factor es $(x - 3)$ (multiplicador $-3$):
   - Multiplicamos $-3 \times 1 = -3$ en la diagonal hacia abajo.
   - Multiplicamos $-3 \times (-2) = 6$ en la diagonal hacia abajo.
   - Sumamos los números en cada columna: $[1, (-2) + (-3), 6] = [1, -5, 6]$, que corresponde a $x^2 - 5x + 6$.
4. El siguiente factor es $(x - 5)$ (multiplicador $-5$):
   - Productos en diagonal: $[-5, 25, -30]$.
   - Suma por columnas: $[1, -10, 31, -30]$, correspondiente a $x^3 - 10x^2 + 31x - 30$.
5. Finalmente, para $(x - 7)$ (multiplicador $-7$):
   - Productos en diagonal: $[-7, 70, -217, 210]$.
   - Suma por columnas: $[1, -17, 101, -247, 210]$.

De esta manera se obtiene el polinomio completo:

$$f(x) = x^4 - 17x^3 + 101x^2 - 247x + 210$$

Este proceso es **cíclico y ordenado**: si almacenamos las $n$ raíces en un arreglo unidimensional, tras un ciclo de $n$ pasos obtenemos el arreglo con los $n+1$ coeficientes del polinomio.


In [2]:
def multiplicacion_sintetica(raices, mostrar_pasos=False):
    """
    Construye los coeficientes de un polinomio mónico a partir de sus raíces
    utilizando el algoritmo de la Multiplicación Sintética.

    Parámetros
    ----------
    raices : list[int | float]
        Arreglo unidimensional con las n raíces del polinomio.
    mostrar_pasos : bool
        Si es True, imprime la tabla paso a paso (diagonales y suma por columnas).

    Retorna
    -------
    list[int | float]
        Coeficientes ordenados de mayor a menor grado.
    """
    coef = [1]

    if mostrar_pasos:
        print(f"Raíces de entrada: {raices}")
        print(f"Coeficiente inicial: {coef}\n")

    for paso, r in enumerate(raices, start=1):
        # Recurrencia: c_j^(k) = c_j^(k-1) - r * c_{j-1}^(k-1)
        nuevo = [coef[0]]
        for j in range(1, len(coef)):
            nuevo.append(coef[j] - r * coef[j - 1])
        nuevo.append(-r * coef[-1])

        if mostrar_pasos:
            ancho = 7
            fila_actual = coef + [0]
            fila_diagonal = [0] + [-r * c for c in coef]
            print(f"Paso {paso} | Factor (x - ({r})) -> multiplicador = {-r}")
            print("  Actual   :", "".join(f"{v:>{ancho}}" for v in fila_actual))
            print("  Diagonal :", "".join(f"{v:>{ancho}}" for v in fila_diagonal))
            print("  " + "-" * (10 + ancho * len(nuevo)))
            print("  Suma     :", "".join(f"{v:>{ancho}}" for v in nuevo))
            print(f"  Polinomio parcial: {formatear_polinomio(nuevo)}\n")

        coef = nuevo

    return coef


raices_ejemplo = [2, 3, 5, 7]
coefs_ejemplo = multiplicacion_sintetica(raices_ejemplo, mostrar_pasos=True)
print(f"Coeficientes finales: {coefs_ejemplo}")
print(f"Polinomio resultante: f(x) = {formatear_polinomio(coefs_ejemplo)}")


Raíces de entrada: [2, 3, 5, 7]
Coeficiente inicial: [1]

Paso 1 | Factor (x - (2)) -> multiplicador = -2
  Actual   :       1      0
  Diagonal :       0     -2
  ------------------------
  Suma     :       1     -2
  Polinomio parcial: x - 2

Paso 2 | Factor (x - (3)) -> multiplicador = -3
  Actual   :       1     -2      0
  Diagonal :       0     -3      6
  -------------------------------
  Suma     :       1     -5      6
  Polinomio parcial: x^2 - 5x + 6

Paso 3 | Factor (x - (5)) -> multiplicador = -5
  Actual   :       1     -5      6      0
  Diagonal :       0     -5     25    -30
  --------------------------------------
  Suma     :       1    -10     31    -30
  Polinomio parcial: x^3 - 10x^2 + 31x - 30

Paso 4 | Factor (x - (7)) -> multiplicador = -7
  Actual   :       1    -10     31    -30      0
  Diagonal :       0     -7     70   -217    210
  ---------------------------------------------
  Suma     :       1    -17    101   -247    210
  Polinomio parcial: x^4 - 17x

## 3. Generación Aleatoria de Polinomios para Cálculo Diferencial

Gracias a la estructura iterativa del algoritmo, generar ejercicios con funciones polinómicas de cualquier grado $g$ y raíces enteras en $[-n, n]$ se reduce a:

1. Seleccionar $g$ valores enteros dentro del intervalo $[-n, n]$.
2. Pasar la lista de raíces a `multiplicacion_sintetica`.

De este modo, el docente dispone de infinitos ejercicios con solución entera garantizada sin importar cuán grande sea el grado del polinomio.


In [3]:
def generar_polinomio_aleatorio(grado, n_rango, permitir_repetidas=False, semilla=None):
    """
    Genera un polinomio mónico aleatorio de grado dado con raíces enteras en [-n_rango, n_rango].
    """
    if grado < 1:
        raise ValueError("El grado debe ser al menos 1.")

    rng = random.Random(semilla)
    universo = list(range(-n_rango, n_rango + 1))

    if permitir_repetidas:
        raices = [rng.choice(universo) for _ in range(grado)]
    else:
        if grado > len(universo):
            raise ValueError("No hay suficientes enteros únicos en el rango solicitado.")
        raices = rng.sample(universo, grado)

    coeficientes = multiplicacion_sintetica(raices)
    return raices, coeficientes


for grado in [2, 3, 4, 5]:
    raices, coefs = generar_polinomio_aleatorio(grado=grado, n_rango=7, semilla=100 + grado)
    print(f"Grado {grado} | Raíces: {raices}")
    print(f"  f(x) = {formatear_polinomio(coefs)}\n")


Grado 2 | Raíces: [-5, 3]
  f(x) = x^2 + 2x - 15

Grado 3 | Raíces: [4, 0, 7]
  f(x) = x^3 - 11x^2 + 28x

Grado 4 | Raíces: [-7, 6, -4, 4]
  f(x) = x^4 + x^3 - 58x^2 - 16x + 672

Grado 5 | Raíces: [7, 2, -2, 6, -6]
  f(x) = x^5 - 7x^4 - 40x^3 + 280x^2 + 144x - 1008



## 4. Evaluación Exacta y Verificación mediante División Sintética

Una propiedad fundamental de esta construcción es su dualidad directa con la **División Sintética** (esquema de Ruffini–Horner). Usando División Sintética podemos:

1. **Evaluar el polinomio numéricamente y de forma exacta** en cualquier valor $x_0$ en tiempo lineal $O(n)$, puesto que el último término obtenido (el residuo $R$) cumple $f(x_0) = R$.
2. **Verificar nuestras raíces y revertir el proceso**: al dividir sucesivamente el polinomio por cada raíz $x = 7, 5, 3, 2$, el residuo es exactamente $0$ en cada etapa y los cocientes recuperan en orden inverso los coeficientes intermedios generados durante la Multiplicación Sintética.


In [4]:
def division_sintetica(coeficientes, x0):
    """
    Aplica División Sintética para dividir el polinomio por (x - x0)
    y evaluar f(x0) de forma exacta.

    Retorna
    -------
    tuple[list[int | float], int | float]
        (coeficientes_cociente, residuo), donde residuo == f(x0).
    """
    cociente = [coeficientes[0]]
    for coef in coeficientes[1:]:
        cociente.append(cociente[-1] * x0 + coef)
    # Por la convención de Horner, el último valor acumulado es precisamente el residuo R = f(x0)
    residuo = cociente.pop()
    return cociente, residuo


print("Verificación por deflación sucesiva sobre f(x) = x^4 - 17x^3 + 101x^2 - 247x + 210:")
actual = list(coefs_ejemplo)
for r in reversed(raices_ejemplo):
    cociente, residuo = division_sintetica(actual, r)
    print(f"  División entre (x - {r}): cociente = {cociente}, residuo f({r}) = {residuo}")
    actual = cociente

print("\nEvaluación exacta de f(x) en otros puntos mediante División Sintética:")
for x_eval in [0, 1, -1, 4, 10]:
    _, valor = division_sintetica(coefs_ejemplo, x_eval)
    valor_directo = (x_eval - 2) * (x_eval - 3) * (x_eval - 5) * (x_eval - 7)
    assert valor == valor_directo
    print(f"  f({x_eval:>2}) = {valor}")


Verificación por deflación sucesiva sobre f(x) = x^4 - 17x^3 + 101x^2 - 247x + 210:
  División entre (x - 7): cociente = [1, -10, 31, -30], residuo f(7) = 0
  División entre (x - 5): cociente = [1, -5, 6], residuo f(5) = 0
  División entre (x - 3): cociente = [1, -2], residuo f(3) = 0
  División entre (x - 2): cociente = [1], residuo f(2) = 0

Evaluación exacta de f(x) en otros puntos mediante División Sintética:
  f( 0) = 210
  f( 1) = 48
  f(-1) = 576
  f( 4) = 6
  f(10) = 840


## 5. Extensión a Factores No Mónicos $(a_k x - b_k)$ y Raíces Múltiples

Para generar polinomios en $\mathbb{Z}[x]$ con **raíces racionales** $r_k = b_k / a_k$ sin introducir coeficientes fraccionarios (y admitiendo raíces de multiplicidad mayor que 1), cada factor lineal $(a_k x - b_k)$ se incorpora mediante la convolución discreta generalizada:

$$c_j^{(k)} = a_k \cdot c_j^{(k-1)} - b_k \cdot c_{j-1}^{(k-1)}, \qquad j = 0, 1, \dots, k$$


In [5]:
def multiplicacion_sintetica_general(factores):
    """
    Construye un polinomio en Z[x] a partir de factores lineales (a*x - b).

    Parámetros
    ----------
    factores : list[tuple[int, int]]
        Pares (a, b) que representan cada factor (a*x - b).
    """
    coef = [1]
    for a, b in factores:
        nuevo = [a * coef[0]]
        for j in range(1, len(coef)):
            nuevo.append(a * coef[j] - b * coef[j - 1])
        nuevo.append(-b * coef[-1])
        coef = nuevo
    return coef


# Ejemplo 1: (2x - 1)(3x + 2) -> factores (2, 1) y (3, -2)
factores_cuad = [(2, 1), (3, -2)]
coefs_cuad = multiplicacion_sintetica_general(factores_cuad)
assert coefs_cuad == [6, 1, -2]
print(f"Factores (2x - 1)(3x + 2): coefs = {coefs_cuad} -> f(x) = {formatear_polinomio(coefs_cuad)}")

# Ejemplo 2: factores racionales y raíz doble x = 1 -> (2x - 1)(3x + 2)(x - 1)^2
factores_mixtos = [(2, 1), (3, -2), (1, 1), (1, 1)]
coefs_mixtos = multiplicacion_sintetica_general(factores_mixtos)
for x_val in [0, 1, 2, -1]:
    esperado = (2 * x_val - 1) * (3 * x_val + 2) * (x_val - 1) ** 2
    _, obtenido = division_sintetica(coefs_mixtos, x_val)
    assert obtenido == esperado
print(f"Polinomio con raíces 1/2, -2/3 y multiplicidad 2 en x=1: f(x) = {formatear_polinomio(coefs_mixtos)}")


Factores (2x - 1)(3x + 2): coefs = [6, 1, -2] -> f(x) = 6x^2 + x - 2
Polinomio con raíces 1/2, -2/3 y multiplicidad 2 en x=1: f(x) = 6x^4 - 11x^3 + 2x^2 + 5x - 2
